In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from zoneinfo import ZoneInfo
import pandas as pd, numpy as np, shutil, json, re, os

ROOT=Path('/content/drive/MyDrive/US_ETF')
CACHE=ROOT/'directional_research/open_revalidation_1m_alpaca_v1/iex'
RES=ROOT/'model_lab_v1/results/open_revalidation_v1'
AUDIT=RES/'open_revalidation_trade_audit.parquet'
STAMP=pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%SZ')
BACKUP=RES/f'open_revalidation_trade_audit.pre_rebuild_{STAMP}.parquet'
REPORT=RES/'open_revalidation_rebuild_report_latest.json'

print('='*100); print('KALMAN OPEN REVALIDATION AUDIT REBUILD v1.6 — RESEARCH ONLY'); print('='*100)
assert CACHE.exists(), f'Missing cache: {CACHE}'
assert AUDIT.exists(), f'Missing audit: {AUDIT}'
df=pd.read_parquet(AUDIT).copy()
print('audit_rows=',len(df),' audit_cols=',len(df.columns))
print('columns=',list(df.columns))

# Resolve canonical identity/timestamp columns conservatively; never invent missing trade times.
def pick(names):
    lower={c.lower():c for c in df.columns}
    for n in names:
        if n.lower() in lower: return lower[n.lower()]
    return None
symcol=pick(['symbol','ticker','asset','asset_symbol'])
entrycol=pick(['entry_time','entry_ts','entry_timestamp','entry_at','entry_datetime','entry_date_time'])
exitcol=pick(['exit_time','exit_ts','exit_timestamp','exit_at','exit_datetime','fixed4_exit_time','fixed4_exit_ts'])
print('\n[RESOLVED SCHEMA] symbol=',symcol,' entry=',entrycol,' exit=',exitcol)
if not symcol or not entrycol or not exitcol:
    print('\n[STOP] Safe rebuild cannot proceed because canonical symbol/entry/exit timestamp columns were not resolved.')
    print('No audit file was modified. Send the columns= output above and the next notebook can bind the exact schema.')
    raise SystemExit(0)

for c in [entrycol,exitcol]: df[c]=pd.to_datetime(df[c],utc=True,errors='coerce')
assert df[entrycol].notna().any() and df[exitcol].notna().any(), 'Resolved timestamps are empty'

# Load Alpaca IEX 1-minute cache once per symbol.
need=set(df[symcol].dropna().astype(str).str.upper())
bars={}; bad=[]
for i,s in enumerate(sorted(need),1):
    fs=sorted((CACHE/s).glob('*.parquet'))
    chunks=[]
    for f in fs:
        try:
            x=pd.read_parquet(f)
            t=next((c for c in ['timestamp','time','datetime','t'] if c in x.columns),None)
            if t is None or 'close' not in x.columns: continue
            x=x.copy(); x['timestamp']=pd.to_datetime(x[t],utc=True,errors='coerce')
            for c in ['open','high','low','close','volume','trade_count','vwap']:
                if c in x.columns: x[c]=pd.to_numeric(x[c],errors='coerce')
            chunks.append(x)
        except Exception as e: bad.append([str(f),str(e)])
    if chunks:
        z=pd.concat(chunks,ignore_index=True).dropna(subset=['timestamp']).sort_values('timestamp')
        z=z.drop_duplicates('timestamp',keep='last').set_index('timestamp')
        bars[s]=z
    if i%10==0 or i==len(need): print(f'[CACHE] {i}/{len(need)} symbols loaded={len(bars)}')

NY=ZoneInfo('America/New_York')
def px_at_or_after(z,t,max_minutes=10,field='close'):
    if z is None or pd.isna(t): return np.nan
    hi=t+pd.Timedelta(minutes=max_minutes)
    q=z.loc[(z.index>=t)&(z.index<=hi)]
    if q.empty or field not in q: return np.nan
    q=q[q[field].notna()]
    return float(q.iloc[0][field]) if len(q) else np.nan
def session_targets(t):
    if pd.isna(t): return (pd.NaT,)*4
    local=t.tz_convert(NY); d=local.date()
    op=pd.Timestamp(year=d.year,month=d.month,day=d.day,hour=9,minute=30,tz=NY).tz_convert('UTC')
    return op,op+pd.Timedelta(minutes=5),op+pd.Timedelta(minutes=15),op
def prev_close(z,op):
    if z is None or pd.isna(op): return np.nan
    q=z.loc[z.index<op]
    if q.empty:return np.nan
    # restrict to prior 7 calendar days; last observed IEX minute is the previous available-session close.
    q=q.loc[q.index>=op-pd.Timedelta(days=7)]
    q=q[q['close'].notna()] if 'close' in q else q.iloc[0:0]
    return float(q.iloc[-1]['close']) if len(q) else np.nan

fields=['entry_price_iex','fixed4_exit_price_iex','prev_close_price_iex','open_0_price_iex','open_5_price_iex','open_15_price_iex']
for c in fields:
    if c not in df.columns: df[c]=np.nan
before={c:int(df[c].isna().sum()) for c in fields}
filled={c:0 for c in fields}

for idx,r in df.iterrows():
    s=str(r[symcol]).upper(); z=bars.get(s); et=r[entrycol]; xt=r[exitcol]
    if z is None: continue
    op,o5,o15,_=session_targets(et)
    vals={
      'entry_price_iex':px_at_or_after(z,et,10,'close'),
      'fixed4_exit_price_iex':px_at_or_after(z,xt,10,'close'),
      'prev_close_price_iex':prev_close(z,op),
      'open_0_price_iex':px_at_or_after(z,op,10,'open'),
      'open_5_price_iex':px_at_or_after(z,o5,10,'close'),
      'open_15_price_iex':px_at_or_after(z,o15,10,'close')}
    for c,v in vals.items():
        if pd.isna(df.at[idx,c]) and pd.notna(v): df.at[idx,c]=v; filled[c]+=1

ready=df[fields].notna().all(axis=1)
df['revalidation_data_ready']=ready
after={c:int(df[c].isna().sum()) for c in fields}
print('\n[REBUILD RESULT]')
print('total_rows =',len(df)); print('ready_rows =',int(ready.sum())); print('not_ready_rows =',int((~ready).sum())); print('ready_rate =',f'{ready.mean():.2%}')
for c in fields: print(f'{c}: missing {before[c]} -> {after[c]} | newly_filled={filled[c]}')

# Safety gates: preserve row count, symbol/time identity, and require strict coverage improvement before replacing canonical audit.
old=pd.read_parquet(AUDIT)
old_ready=int(old['revalidation_data_ready'].fillna(False).astype(bool).sum()) if 'revalidation_data_ready' in old else 0
new_ready=int(ready.sum())
identity_ok=(len(old)==len(df) and old[symcol].astype(str).tolist()==df[symcol].astype(str).tolist())
improved=new_ready>old_ready
report={'schema':'kalman-open-revalidation-audit-rebuild-v1.6','research_only':True,'production_changed':False,'rows':len(df),'old_ready':old_ready,'new_ready':new_ready,'ready_rate':float(ready.mean()),'before_missing':before,'after_missing':after,'newly_filled':filled,'identity_ok':bool(identity_ok),'coverage_improved':bool(improved),'cache_symbols_loaded':len(bars),'cache_read_errors':len(bad)}
REPORT.write_text(json.dumps(report,indent=2),encoding='utf-8')
if identity_ok and improved:
    shutil.copy2(AUDIT,BACKUP)
    tmp=AUDIT.with_suffix('.tmp.parquet'); df.to_parquet(tmp,index=False); os.replace(tmp,AUDIT)
    print('\n[WRITE] PASS — canonical audit replaced atomically')
    print('backup =',BACKUP); print('audit =',AUDIT)
else:
    cand=RES/'open_revalidation_trade_audit.rebuild_candidate.parquet'; df.to_parquet(cand,index=False)
    print('\n[WRITE] BLOCKED — canonical audit NOT modified')
    print('candidate =',cand,' identity_ok=',identity_ok,' improved=',improved)
print('report =',REPORT)
print('\nNEXT: rerun Kalman_System_Checklist_Master_v1_5_Open_Revalidation_Colab.ipynb. R9 promotion remains blocked pending prospective/fold/bootstrap gates.')
